In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Gifts/Travel or Lodging Expenses_Non POs
   
   BUSINESS QUESTION: 
   During the assessment period, how many instances were recorded by the Unit where 
   employees provide or receive gifts or paid for travel/entertainment to Non POs 
   exceeding the $250 threshold?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. COUPA DATA ONLY: Unions the 7 Coupa files, pulling the 'Total' column.
   3. ACCOUNT PARSING: Splits the Account string to extract the Cost Center (index 0) 
      and the Category Code (index 2).
   4. THRESHOLD RULE: Normalizes all amount-like tokens found in `Total`, uses the
      maximum parsed amount, then enforces Numeric_Total > 250.
   5. NON-PUBLIC OFFICIAL FILTER: Strictly filters for 'N' or 'NO'.
   6. TARGET CATEGORY CODES: Hardcoded list of 29 specific category codes.
   7. THE 793 EXCEPTION RULE: If the Category Code is '793', it is IGNORED unless 
      the mapped AU is strictly '101016' (TD Asset Management).
   8. FINAL OUTPUT: Returns '100%' when at least one matching transaction exists 
      for the AU and '0%' otherwise, anchored to the Master List.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    -- 2. Union all 7 Coupa files into one master dataset, including the Total column
    SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3 & 4. Parse the Account string and clean the Total for mathematical comparison
    SELECT 
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        aggregate(
            filter(
                transform(
                    filter(
                        split(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(Total, ''), '[^0-9,\\.]', ' '), '\\s+', ' ')), ' '),
                        token -> token RLIKE '.*[0-9].*'
                    ),
                    token -> TRY_CAST(
                        CASE
                            WHEN token RLIKE '^[0-9]+,[0-9]{2}$' THEN REPLACE(token, ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$' THEN REPLACE(REPLACE(token, '.', ''), ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(,[0-9]{3})+\\.[0-9]{2}$' THEN REPLACE(token, ',', '')
                            WHEN token RLIKE '^[0-9]+(\\.[0-9]{2})?$' THEN token
                            ELSE NULL
                        END AS DECIMAL(10,2)
                    )
                ),
                amount -> amount IS NOT NULL
            ),
            CAST(NULL AS DECIMAL(10,2)),
            (acc, amount) -> CASE WHEN acc IS NULL OR amount > acc THEN amount ELSE acc END
        ) AS Numeric_Total
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    -- Standardize the CC Mapping table from our bootstrap
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_Transactions AS (
    -- 5, 6, 7. Apply all Business Rules to filter valid transactions
    SELECT 
        c.Cleaned_CC,
        c.CatCode,
        m.AU_ID
    FROM Coupa_Parsed c
    INNER JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- RULE: Must NOT be a Public Official
        UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO')
        
        -- RULE: Threshold must be strictly greater than 250
        AND c.Numeric_Total > 250
        
        -- RULE: Must be one of the target category codes
        AND c.CatCode IN (
            '066', '009', '012', '067', '095', '134', '168', '192', '203', '204', 
            '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', 
            '639', '676', '782', '783', '784', '792', '793', '890', '892'
        )
        
        -- RULE: The 793 Exception Rule (Ignores 793 unless the mapped AU is 101016)
        AND (c.CatCode != '793' OR (c.CatCode = '793' AND m.AU_ID = '101016'))
),

Flagged_AUs AS (
    -- Map flagged transactions to their AU_IDs and COUNT them
    SELECT 
        AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Transactions 
    GROUP BY AU_ID
)

-- 8. Final Output: Strictly structured to 6 columns, anchored to the Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA04' AS QuestionID,               
    CASE WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN '100%' ELSE '0%' END AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA04 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final percentage
   was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Debug_Base AS (
    SELECT 
        Account AS Raw_Account,
        CASE 
            WHEN Account LIKE '%-%-%' THEN 'Pass'
            ELSE 'FAIL: Account not in expected hyphenated format'
        END AS Account_Format_Check,
        TRIM(SPLIT(Account, '-')[0]) AS Raw_Account_CC_Block,
        CASE 
            WHEN Account LIKE '%-%-%' THEN CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END
            ELSE NULL
        END AS Cleaned_CC,
        CASE 
            WHEN Account LIKE '%-%-%' THEN TRIM(SPLIT(Account, '-')[2])
            ELSE NULL
        END AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial,
        Total AS Raw_Total_String,
        aggregate(
            filter(
                transform(
                    filter(
                        split(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(COALESCE(Total, ''), '[^0-9,\\.]', ' '), '\\s+', ' ')), ' '),
                        token -> token RLIKE '.*[0-9].*'
                    ),
                    token -> TRY_CAST(
                        CASE
                            WHEN token RLIKE '^[0-9]+,[0-9]{2}$' THEN REPLACE(token, ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$' THEN REPLACE(REPLACE(token, '.', ''), ',', '.')
                            WHEN token RLIKE '^[0-9]{1,3}(,[0-9]{3})+\\.[0-9]{2}$' THEN REPLACE(token, ',', '')
                            WHEN token RLIKE '^[0-9]+(\\.[0-9]{2})?$' THEN token
                            ELSE NULL
                        END AS DECIMAL(10,2)
                    )
                ),
                amount -> amount IS NOT NULL
            ),
            CAST(NULL AS DECIMAL(10,2)),
            (acc, amount) -> CASE WHEN acc IS NULL OR amount > acc THEN amount ELSE acc END
        ) AS Numeric_Total
    FROM Combined_Coupa_Raw
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Evaluated_Rows AS (
    SELECT 
        c.Cleaned_CC,
        c.CatCode,
        c.PublicOfficial,
        c.Numeric_Total,
        m.AU_ID AS Mapped_AU_ID,
        mast.BusinessID AS Master_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        mast.In_ABAC_AU_List,
        CASE WHEN c.Account_Format_Check = 'Pass' THEN 1 ELSE 0 END AS Parse_Pass,
        CASE WHEN m.AU_ID IS NOT NULL AND mast.BusinessID IS NOT NULL THEN 1 ELSE 0 END AS Bridge_Pass,
        CASE WHEN UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO') THEN 1 ELSE 0 END AS Non_PO_Pass,
        CASE WHEN c.Numeric_Total IS NOT NULL AND c.Numeric_Total > 250 THEN 1 ELSE 0 END AS Threshold_Pass,
        CASE WHEN c.CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892') THEN 1 ELSE 0 END AS CatCode_Pass,
        CASE WHEN c.CatCode = '793' AND mast.BusinessID != '101016' THEN 0 ELSE 1 END AS Exception_793_Pass,
        CASE 
            WHEN c.Account_Format_Check = 'Pass'
             AND mast.BusinessID IS NOT NULL
             AND UPPER(TRIM(c.PublicOfficial)) IN ('N', 'NO')
             AND c.Numeric_Total IS NOT NULL
             AND c.Numeric_Total > 250
             AND c.CatCode IN ('066', '009', '012', '067', '095', '134', '168', '192', '203', '204', '208', '209', '242', '269', '270', '484', '487', '636', '637', '638', '639', '676', '782', '783', '784', '792', '793', '890', '892')
             AND NOT (c.CatCode = '793' AND mast.BusinessID != '101016')
            THEN 1 ELSE 0
        END AS Counted_Flag
    FROM Coupa_Debug_Base c
    LEFT JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
    LEFT JOIN Master_AUs mast
        ON m.AU_ID = mast.BusinessID
),

AU_Debug AS (
    SELECT 
        Master_AU_ID AS AU_ID,
        MAX(AU_Name) AS AU_Name,
        MAX(Business_Segment) AS Business_Segment,
        MAX(In_ABAC_AU_List) AS In_ABAC_AU_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN Counted_Flag = 1 THEN Cleaned_CC END))) AS Counted_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN Counted_Flag = 1 THEN CatCode END))) AS Counted_CatCode_List,
        COUNT(*) AS Mapped_Row_Count,
        SUM(Non_PO_Pass) AS Non_PO_Row_Count,
        SUM(Threshold_Pass) AS Above_Threshold_Row_Count,
        SUM(CatCode_Pass) AS Valid_CatCode_Row_Count,
        SUM(Exception_793_Pass) AS Exception_793_Pass_Row_Count,
        SUM(Counted_Flag) AS Counted_Row_Count
    FROM Evaluated_Rows
    WHERE Master_AU_ID IS NOT NULL
    GROUP BY Master_AU_ID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA04' AS QuestionID,
    CASE WHEN COALESCE(d.Counted_Row_Count, 0) > 0 THEN '100%' ELSE '0%' END AS Response,
    COALESCE(d.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(d.Counted_CC_List, 'n/a') AS Counted_CC_List,
    COALESCE(d.Counted_CatCode_List, 'n/a') AS Counted_CatCode_List,
    COALESCE(d.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(d.Non_PO_Row_Count, 0) AS Non_PO_Row_Count,
    COALESCE(d.Above_Threshold_Row_Count, 0) AS Above_Threshold_Row_Count,
    COALESCE(d.Valid_CatCode_Row_Count, 0) AS Valid_CatCode_Row_Count,
    COALESCE(d.Exception_793_Pass_Row_Count, 0) AS Exception_793_Pass_Row_Count,
    COALESCE(d.Counted_Row_Count, 0) AS Counted_Row_Count,
    CASE WHEN COALESCE(d.Counted_Row_Count, 0) > 0 THEN 1 ELSE 0 END AS Numerator,
    1 AS Denominator,
    CASE WHEN COALESCE(d.Counted_Row_Count, 0) > 0 THEN '100%' ELSE '0%' END AS Calculated_Percentage,
    CASE 
        WHEN COALESCE(d.Counted_Row_Count, 0) > 0 THEN CONCAT(
            'Response = 100% because 1 qualifying outcome / 1 total outcome. ',
            CAST(d.Counted_Row_Count AS STRING),
            ' counted row(s) from ', CAST(d.Mapped_Row_Count AS STRING), ' mapped row(s). Non-PO=',
            CAST(d.Non_PO_Row_Count AS STRING), ', >250=', CAST(d.Above_Threshold_Row_Count AS STRING),
            ', valid CatCode=', CAST(d.Valid_CatCode_Row_Count AS STRING), ', 793-pass=',
            CAST(d.Exception_793_Pass_Row_Count AS STRING)
        )
        WHEN COALESCE(d.Mapped_Row_Count, 0) > 0 THEN CONCAT(
            'Response = 0% because 0 qualifying outcomes / 1 total outcome. 0 counted row(s) from ', CAST(d.Mapped_Row_Count AS STRING), ' mapped row(s). Non-PO=',
            CAST(d.Non_PO_Row_Count AS STRING), ', >250=', CAST(d.Above_Threshold_Row_Count AS STRING),
            ', valid CatCode=', CAST(d.Valid_CatCode_Row_Count AS STRING), ', 793-pass=',
            CAST(d.Exception_793_Pass_Row_Count AS STRING)
        )
        ELSE 'Response = 0% because 0 qualifying outcomes / 1 total outcome. No source rows bridged to this AU.'
    END AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.AU_ID
ORDER BY mast.BusinessID;